# Lab 3: Explore bakery sales with SQL and Polars

**Goal:** use the same bakery data to answer questions in SQL and in Polars. For each question, run both versions and check that they return the same columns, rows, values, and order. Turn in what you finish; the later questions give you more to explore if time allows.

The [Kaggle bakery dataset](https://www.kaggle.com/datasets/sdcloud/historical-bakery-sales-dataset) is **synthetic**. It is useful for query practice, but any pattern you find is not evidence about real bakery customers. This notebook downloads version 2 of the CSV with [KaggleHub](https://github.com/Kaggle/kagglehub). Internet access is needed on the first run; later runs can use KaggleHub's cache.

## 0. Load one dataset for both query styles

Run the next two cells. The first installs missing packages or upgrades Polars to version 1.41 or newer (needed for SQL aggregate `FILTER`), then downloads the CSV. If it upgrades a package that was already imported, restart the notebook kernel and run the cells again. Public Kaggle datasets usually download without a login. If Kaggle asks for authorization, use your own Kaggle account or ask the instructor for a copy of the CSV; do not put an API token in this notebook.

We keep seven columns from the 40-column source. `bakery` is the SQL table name for the same Polars DataFrame called `sales`. **Polars SQL runs on the DataFrame; no PostgreSQL server is needed.**

In [ ]:
import importlib.util
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys


def major_minor(version_text):
    major, minor = version_text.split(".")[:2]
    return int(major), int(minor)


try:
    installed_polars = version("polars")
except PackageNotFoundError:
    installed_polars = None

needed = []
if installed_polars is None or major_minor(installed_polars) < (1, 41):
    needed.append("polars>=1.41.0")
if importlib.util.find_spec("kagglehub") is None:
    needed.append("kagglehub")
if needed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *needed])

import kagglehub
import polars as pl

if major_minor(pl.__version__) < (1, 41):
    raise RuntimeError(
        f"This kernel still has Polars {pl.__version__} loaded. "
        "Restart the kernel and run the notebook from the top."
    )
print(f"Polars version: {pl.__version__}")

csv_path = kagglehub.dataset_download(
    "sdcloud/historical-bakery-sales-dataset/versions/2",
    path="bakery_sales_dataset.csv",
)
print(f"CSV location: {csv_path}")

In [ ]:
columns = [
    "Order_ID", "Date", "Product", "Category", "Quantity",
    "Discount_Percentage", "Store_ID",
]
sales = pl.read_csv(csv_path, columns=columns)
ctx = pl.SQLContext(bakery=sales, eager=True)

print(f"Rows: {sales.height:,}; columns: {sales.width}")
print(sales.schema)
sales.head(5)

### First check: What does a row represent?

`Order_ID` is repeated when an order contains multiple product lines. Compare the number of rows with the number of distinct order IDs. The two queries below should agree. A later `COUNT(*)` will count **lines**, not whole orders.

In [ ]:
ctx.execute("""
SELECT COUNT(*) AS lines, COUNT(DISTINCT Order_ID) AS orders
FROM bakery
""")

In [ ]:
sales.select(
    pl.len().alias("lines"),
    pl.col("Order_ID").n_unique().alias("orders"),
)

## 1. Filter, select, and order

**Concept:** SQL uses `WHERE`; Polars uses `.filter(...)`. SQL `SELECT` corresponds to `.select(...)`, `ORDER BY` to `.sort(...)`, and `LIMIT 10` to `.head(10)`. In the examples below, sort the larger quantities first; use order ID and product name to make ties reproducible.

In [ ]:
ctx.execute("""
SELECT Order_ID, Product, Quantity
FROM bakery
WHERE Quantity >= 5
ORDER BY Quantity DESC, Order_ID, Product
LIMIT 10
""")

In [ ]:
(
    sales.filter(pl.col("Quantity") >= 5)
    .select("Order_ID", "Product", "Quantity")
    .sort("Quantity", "Order_ID", "Product", descending=[True, False, False])
    .head(10)
)

### Your turn 1 — Which lines had the largest recorded discounts?

Find **ten product lines** with the largest positive `Discount_Percentage` in the bakery data.

1. Keep rows where `Discount_Percentage > 0`.
2. Show `Order_ID`, `Product`, `Category`, and `Discount_Percentage`.
3. Sort by discount percentage **largest first**. Break ties by `Order_ID`, then `Product`, both A–Z.
4. Return the first ten rows. Write the query in SQL and Polars, then compare the results.

**Check:** The SQL and Polars results should show the same ten lines in the same order.

In [ ]:
# SQL for your turn 1

In [ ]:
# Polars for your turn 1

## 2. Group and summarize

**Concept:** `GROUP BY` and `.group_by(...).agg(...)` collapse many product lines into one row per group.

- `COUNT(*)` and `pl.len()` count **lines** (rows).
- `SUM(Quantity)` and `pl.col("Quantity").sum()` count **units** (individual bakery items listed on those lines).

For example, two lines with `Quantity` values 3 and 5 are **2 lines** and **8 units**. `units` is an output name we give to `SUM(Quantity)`, not a column already present in this lab's DataFrame. Start by grouping by product category.

In [ ]:
ctx.execute("""
SELECT Category, COUNT(*) AS lines, SUM(Quantity) AS units
FROM bakery
GROUP BY Category
ORDER BY Category
""")

In [ ]:
(
    sales.group_by("Category")
    .agg(
        pl.len().alias("lines"),
        pl.col("Quantity").sum().alias("units"),
    )
    .sort("Category")
)

### Your turn 2 — How do the stores compare?

For each `Store_ID`, report two totals:

1. **`lines`:** the number of product-line rows for that store (`COUNT(*)` or `pl.len()`).
2. **`units`:** the sum of `Quantity` across those rows—the number of individual bakery items recorded as purchased at that store.

Sort by `units`, largest first. Write both versions. Do not count distinct products or orders for this question.

**Check:** Both results should have one row per store, with matching `lines` and `units` values.

In [ ]:
# SQL for your turn 2

In [ ]:
# Polars for your turn 2

## 3. Filter before grouping

**Concept:** Apply a row filter first, then group the retained rows. This summary counts discounted lines and their units by category. The filter changes which lines enter the summary; it does not filter completed groups.

In [ ]:
ctx.execute("""
SELECT Category, COUNT(*) AS discounted_lines, SUM(Quantity) AS discounted_units
FROM bakery
WHERE Discount_Percentage > 0
GROUP BY Category
ORDER BY discounted_units DESC, Category
""")

In [ ]:
(
    sales.filter(pl.col("Discount_Percentage") > 0)
    .group_by("Category")
    .agg(
        pl.len().alias("discounted_lines"),
        pl.col("Quantity").sum().alias("discounted_units"),
    )
    .sort("discounted_units", "Category", descending=[True, False])
)

### Your turn 3 — Which discounted products account for the most units?

Among discounted lines, show the **five products** with the greatest total quantity. Include each product's line count and unit count. Sort by units descending, then product name to break ties.

**Check:** Both results should show the same five products, counts, and order.

In [ ]:
# SQL for your turn 3

In [ ]:
# Polars for your turn 3

## 4. Filter one aggregate, not every row

**Concept:** SQL `WHERE` removes rows before **all** aggregates. `FILTER (WHERE ...)` limits **one aggregate**, so the same grouped result can show counts for all lines and a subset. In Polars, `.filter(...)` can be applied to an expression inside `.agg(...)`.

The worked example counts all product lines and discounted lines within each category. `Order_ID` is present on every line, so counting filtered `Order_ID` values in Polars counts the retained lines.

In [ ]:
category_summary_sql = ctx.execute("""
SELECT Category,
       COUNT(*) AS lines,
       COUNT(*) FILTER (WHERE Discount_Percentage > 0) AS discounted_lines
FROM bakery
GROUP BY Category
ORDER BY Category
""")
category_summary_sql

In [ ]:
category_summary_pl = (
    sales.group_by("Category")
    .agg(
        pl.len().alias("lines"),
        pl.col("Order_ID")
          .filter(pl.col("Discount_Percentage") > 0)
          .count()
          .alias("discounted_lines"),
    )
    .sort("Category")
)
category_summary_pl

### Your turn 4 — Which stores have more large-quantity lines?

For each `Store_ID`, report **all product lines** as `lines` and lines with `Quantity >= 5` as `large_lines`. Use `COUNT(*) FILTER (WHERE ...)` in SQL and an expression-level `.filter(...)` in Polars. Sort by `Store_ID`. Save the results as `store_summary_sql` and `store_summary_pl`; you will use both in the next question.

**Check:** Each store should have `large_lines <= lines`, and the SQL and Polars counts should match.

In [ ]:
# SQL for your turn 4: assign the result to store_summary_sql


In [ ]:
# Polars for your turn 4: assign the result to store_summary_pl


## 5. Query the result of a previous query

**Concept:** A named result can become the input to another query. In a notebook, `ctx.register(name, frame)` makes a Polars DataFrame available under a SQL table name for later `ctx.execute(...)` calls. This registration lasts only in the current Python session; it does not create a PostgreSQL table or view. In the Polars expression style, use the named DataFrame variable directly.

Here we reuse the category summary from the worked example in part 4. The new calculation is the **percentage of lines with a discount**, not the average discount amount.

In [ ]:
ctx.register("category_summary", category_summary_sql)
ctx.execute("""
SELECT Category, lines, discounted_lines,
       ROUND(100.0 * discounted_lines / lines, 1) AS pct_discounted
FROM category_summary
ORDER BY pct_discounted DESC, Category
""")

In [ ]:
(
    category_summary_pl.with_columns(
        (100.0 * pl.col("discounted_lines") / pl.col("lines"))
        .round(1)
        .alias("pct_discounted")
    )
    .sort("pct_discounted", "Category", descending=[True, False])
)

### Your turn 5 — What share of each store's lines are large?

Use the two **store summaries you created in part 4**. For each store, calculate `pct_large = 100 × large_lines / lines`, rounded to one decimal place, and sort from highest percentage to lowest.

- **SQL:** register `store_summary_sql` in `ctx` as `store_summary`, then run a new `SELECT` from `store_summary`.
- **Polars:** start from `store_summary_pl`; add `pct_large` with `.with_columns(...)`, then sort.

Do not group the original bakery rows again. Check that both results have the same stores, percentages, and order.

In [ ]:
# SQL for your turn 5: register store_summary_sql, then query store_summary


In [ ]:
# Polars for your turn 5: extend store_summary_pl


## Before you turn it in

For each question you completed, compare the SQL and Polars results: columns, row count, values, and order should match. If they differ, check the filter, grouping key, and sort order. Turn in the notebook with what you finish.

**References:** [Polars SQL](https://docs.pola.rs/user-guide/sql/intro/), [Polars aggregation](https://docs.pola.rs/user-guide/expressions/aggregation/), and [KaggleHub downloads](https://github.com/Kaggle/kagglehub#download-dataset). Dataset credit: [sdcloud on Kaggle](https://www.kaggle.com/datasets/sdcloud/historical-bakery-sales-dataset), CC0 Public Domain.